In [1]:
!pip install fastmcp

  Using cached mcp-1.26.0-py3-none-any.whl.metadata (89 kB)
Using cached mcp-1.26.0-py3-none-any.whl (233 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 1.23.3
    Uninstalling mcp-1.23.3:
      Successfully uninstalled mcp-1.23.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.9.3 requires mcp~=1.23.1, but you have mcp 1.26.0 which is incompatible.
crewai 1.9.3 requires tokenizers~=0.20.3, but you have tokenizers 0.22.2 which is incompatible.


In [1]:
my_code = "hello"
print(my_code)

hello


In [2]:
import sys
sys.stdout.write("Hello")

Hello

5

In [3]:
name = input("What is your name?")

sys.exit("Notebook exited after input.")


What is your name? ani


SystemExit: Notebook exited after input.

/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
sys.stdout.write(name)

ani

3

In [5]:
# Normal output goes to stdout
print("This is standard output (stdout)")

# Error messages can be written to stderr
print("This is an error message (stderr)", file=sys.stderr)

# Example: catching a real error
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error occurred: {e}", file=sys.stderr)

This is standard output (stdout)


This is an error message (stderr)
Error occurred: division by zero


In [6]:
from fastmcp import Client
from fastmcp.client.transports import StdioTransport,StreamableHttpTransport

In [7]:
stdio_transport = StdioTransport(
    command = "npx",
    args = ["-y", "@upstash/context7-mcp"]
)
print(stdio_transport)

<StdioTransport(command='npx', args=['-y', '@upstash/context7-mcp'])>


In [8]:
stdio_client = Client(stdio_transport)

In [9]:
async with stdio_client as client:
    tools = await client.list_tools()
print("Done")    

Done


In [10]:
len(tools)

2

In [11]:
tools

[Tool(name='resolve-library-id', title='Resolve Context7 Library ID', description="Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.\n\nYou MUST call this function before 'query-docs' to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.\n\nSelection Process:\n1. Analyze the query to understand what library/package the user is looking for\n2. Return the most relevant match based on:\n- Name similarity to the query (exact matches prioritized)\n- Description relevance to the query's intent\n- Documentation coverage (prioritize libraries with higher Code Snippet counts)\n- Source reputation (consider libraries with High or Medium reputation more authoritative)\n- Benchmark Score: Quality indicator (100 is the highest score)\n\nResponse Format:\n- Return the selected library ID in a clearly marked section\n- Provide a brief exp

In [12]:
print(tools[0].name)

resolve-library-id


In [13]:
print(tools[0].description)

Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'query-docs' to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Selection Process:
1. Analyze the query to understand what library/package the user is looking for
2. Return the most relevant match based on:
- Name similarity to the query (exact matches prioritized)
- Description relevance to the query's intent
- Documentation coverage (prioritize libraries with higher Code Snippet counts)
- Source reputation (consider libraries with High or Medium reputation more authoritative)
- Benchmark Score: Quality indicator (100 is the highest score)

Response Format:
- Return the selected library ID in a clearly marked section
- Provide a brief explanation for why this library was chosen
- If multiple good matches exist, acknowledge this but pr

In [14]:
tools[0].inputSchema

{'$schema': 'http://json-schema.org/draft-07/schema#',
 'type': 'object',
 'properties': {'query': {'type': 'string',
   'description': "The user's original question or task. This is used to rank library results by relevance to what the user is trying to accomplish. IMPORTANT: Do not include any sensitive or confidential information such as API keys, passwords, credentials, or personal data in your query."},
  'libraryName': {'type': 'string',
   'description': 'Library name to search for and retrieve a Context7-compatible library ID.'}},
 'required': ['query', 'libraryName']}

In [15]:
print(
f""" name: {tools[1].name}: \n
description: {tools[1].description} \n
inputSchema: {tools[1].inputSchema}""")

 name: query-docs: 

description: Retrieves and queries up-to-date documentation and code examples from Context7 for any programming library or framework.

You must call 'resolve-library-id' first to obtain the exact Context7-compatible library ID required to use this tool, UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

IMPORTANT: Do not call this tool more than 3 times per question. If you cannot find what you need after 3 calls, use the best information you have. 

inputSchema: {'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'libraryId': {'type': 'string', 'description': "Exact Context7-compatible library ID (e.g., '/mongodb/docs', '/vercel/next.js', '/supabase/supabase', '/vercel/next.js/v14.3.0-canary.87') retrieved from 'resolve-library-id' or directly from user query in the format '/org/project' or '/org/project/version'."}, 'query': {'type': 'string', 'description': 

In [16]:
async with stdio_client as client:
     # Find a library ID via a search query
    response = await client.call_tool(
        "resolve-library-id", {
            "libraryName" : "fastmcp",
            "query": "How to create a new mcp server with fastmcp python framework?."
        }
    )
print(response.content[0].text)    

Available Libraries:

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score, and relevance to your use case.

----------

- Title: FastMCP
- Context7-compatible library ID: /jlowin/fastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers and clients, simplifying the creation of LLM-integrated applications with features like deployment, authentication, and dynamic tool generation.
- Code Sn

In [17]:
async with stdio_client as client:
    docs = await client.call_tool(
        "query-docs",{
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
        }
    )
print(docs.content[0].text[:1000])    

### GitHub Code Search Syntax Guide

Source: https://github.com/permitio/permit-fastmcp/blob/main/docs/configuration-reference

This snippet provides a link to the official GitHub documentation for code search syntax. It's a crucial resource for users wanting to leverage the full capabilities of GitHub's code search engine.

```HTML
<a target="_blank" href="https://docs.github.com/search-github/github-code-search/understanding-github-code-search-syntax" data-view-component="true" class="Link color-fg-accent text-normal ml-2">Search syntax tips</a>
```

--------------------------------

### GitHub Code Search Documentation Link

Source: https://github.com/permitio/permit-fastmcp/blob/main/docs/advanced-configuration

This snippet displays a paragraph with a link to the official GitHub documentation for understanding code search syntax. It guides users on how to find all available qualifiers.

```html
<p class="text-small color-fg-muted">
            To see all available qualifiers, see 

In [18]:
import requests

In [19]:
url = "https://www.ibm.com/"
r = requests.get(url)
r.content

b' \n<!DOCTYPE HTML>\n\n<html lang="en">\n<head>\r\n    \r\n    \r\n    \r\n    \r\n    \r\n    \r\n    \r\n      \r\n    \r\n    \r\n    \r\n    \r\n    <meta charset="UTF-8"/>\r\n    <meta name="languageCode" content="en"/>\r\n    <meta name="countryCode" content="us"/>\r\n    <meta name="searchTitle" content="IBM"/>\r\n    <meta name="focusArea" content="Cross IBM SDRs"/>\r\n    <meta name=\'primaryTaxonomyEn\' content="IBM"/>\r\n    <meta name="siteSection" content="home"/>\r\n    <meta name="primaryTopic" content="IBM"/>\r\n    <meta name="productName"/>\r\n    <meta name="pageType"/>\r\n    <title>IBM</title>\r\n      <meta name="content-page-ref" content="eROxip4tByGTXXkYPHZou7oTgw2e0GMdApm2VHP9JNKXdpCWq8mPAxUAd36QHdh0uX5DLAELN8kcObodQWtduQ"/>\n<script defer="defer" type="text/javascript" src="https://rum.hlx.page/.rum/@adobe/helix-rum-js@%5E2/dist/rum-standalone.js" data-routing="program=131558,environment=1281329,tier=publish"></script>\n<link rel="icon" sizes="16x16" href="/c

In [20]:
http_transport = StreamableHttpTransport(
    url = "https://mcp.context7.com/mcp"
)

In [21]:
http_client = Client(http_transport)

In [22]:
async with http_client as client:
    tools = await client.list_tools()

    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

In [24]:
for tool in tools:
        print(
f"""{tool.name}: \n
{tool.description} \n
{tool.inputSchema}""")
print(response.content[0].text[:1000])
print(docs.content[0].text[:500]) 

resolve-library-id: 

Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'query-docs' to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Selection Process:
1. Analyze the query to understand what library/package the user is looking for
2. Return the most relevant match based on:
- Name similarity to the query (exact matches prioritized)
- Description relevance to the query's intent
- Documentation coverage (prioritize libraries with higher Code Snippet counts)
- Source reputation (consider libraries with High or Medium reputation more authoritative)
- Benchmark Score: Quality indicator (100 is the highest score)

Response Format:
- Return the selected library ID in a clearly marked section
- Provide a brief explanation for why this library was chosen
- If multiple good matches exist, a